# PhysAR-IDS -- NB3 (v6): Attack 1 -- grey-box surrogate-transfer FGSM / PGD -- multiclass, multiseed

**Target (defender):** the **Optuna-tuned XGBoost** from NB2 v6 (`xgb_tuned_best_params_v6.json`), refitted for each of the 5 seeds on the same GPU backend as NB2.
The **untuned baseline XGBoost** (same backend, same seeds) is attacked with the *same* adversarial examples as a secondary row, so the tuned/untuned choice can be checked against the attack finding.

**Threat model (stated explicitly, per the reviewer).** *Grey-box transfer*: the attacker knows the feature space and has labelled training-like data, trains a
surrogate MLP on it, crafts adversarial examples on the **surrogate's gradients**, and submits them to the target. The target's parameters and gradients are never used.
(This is not a white-box attack and should not be called one.) Perturbations are L-infinity bounded by epsilon in **standardised feature units (sigma)**, and are
**not** projected onto physically realisable values.

**What is new relative to v4**

| Change | Reason |
|---|---|
| 93-feature v6 inputs, tuned target loaded from NB2 v6 | v4 numbers rest on the corrupted v4 pipeline |
| **5 seeds for everything** (surrogate, target, attack), mean +/- std, 95% t-interval | v4 reported a single run plus a surrogate-only seed check |
| **Feature-group threat models**: all features / **cyber-only** (network-observable, 60 cols) / **physical-only** (33 cols) | Reviewer: "physics immunity" must be tied to what an attacker can actually modify. A network attacker controls traffic features, not the sensor values |
| Constant features reported | 7 of the 93 features are constant; perturbing them is physically meaningless but is allowed in the "all" setting |
| **Burst-clustered bootstrap CI** next to the row-level one | Rows come in ~229-row identical bursts; a row-level bootstrap gives an absurdly narrow CI (v4: width 0.006) |
| Paired tests across seeds (PGD vs FGSM, tuned vs baseline target) | v4 only counted wins |
| Surrogate state dicts for all seeds + eps=0.1 adversarial set saved | needed by Defence 1 (NB5) |

Inputs: `phase1_data_v6.pkl`, `xgb_tuned_best_params_v6.json`, `optuna_xgboost_results_v6.pkl`. Outputs are listed at the end.

## 0. Setup

In [ ]:
import os, sys, random, time, json, platform, warnings
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score, classification_report
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS      = [42, 7, 123, 2024, 99]
EPSILONS   = [0.0, 0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]
EPS_DETAIL = 0.1
PGD_ITERS  = 10
SMOKE_TEST = False

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} on {DEVICE} | xgboost {xgb.__version__} backend: {BACKEND_NAME}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX'])
PHYS_IDX = np.setdiff1d(np.arange(len(FEATURE_COLS)), CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
N_FEAT = X_train.shape[1]
const_idx = np.where(X_train.std(0) < 1e-9)[0]
print(f'Train {X_train.shape}  Test {X_test.shape} | cyber features {len(CYBER_IDX)}, physical/derived {len(PHYS_IDX)}, constant {len(const_idx)}')
print('Constant features:', [FEATURE_COLS[i] for i in const_idx])

# bursts (class, PLC, timestamp) in the test set, for the clustered bootstrap
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0]
print(f'Test rows {len(y_test)} in {test_burst.max() + 1} bursts')

best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
BASELINE_PARAMS = dict(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8)
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)
print('Tuned params:', best_params)
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 recorded tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Surrogate and attacks

* `SurrogateMLP` (93-64-32-6), trained 20 epochs with Adam on the **true labels** (not the target's predictions) -- same recipe as v4.
* FGSM: one signed-gradient step of size eps. PGD: 10 steps of size eps/4, projected to the eps-ball after each step.
* `mask` restricts which features may be perturbed (all / cyber-only / physical-only). The perturbation budget is per feature, in standardised units.
* No clipping to the observed feature range: the attack is deliberately unconstrained (stated limitation; a feature-range-constrained variant belongs with Attack 2/3).

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)

def train_surrogate(seed, X_tr, y_tr, n_epochs=20):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed); torch.cuda.manual_seed_all(seed)
    model = SurrogateMLP(X_tr.shape[1]).to(DEVICE); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    ds = TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).long())
    loader = DataLoader(ds, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed))
    for ep in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
    model.eval(); return model

def _mask_t(mask):
    m = np.ones(N_FEAT, dtype=np.float32) if mask is None else mask.astype(np.float32)
    return torch.from_numpy(m).to(DEVICE)

def fgsm_attack(model, X, y, eps, mask=None):
    model.eval(); m = _mask_t(mask)
    Xt = torch.from_numpy(X).float().to(DEVICE).requires_grad_(True); yt = torch.from_numpy(y).long().to(DEVICE)
    F.cross_entropy(model(Xt), yt).backward()
    return (Xt + eps * m * Xt.grad.sign()).detach().cpu().numpy()

def pgd_attack(model, X, y, eps, mask=None, n_iter=PGD_ITERS):
    model.eval(); m = _mask_t(mask); alpha = eps / 4
    yt = torch.from_numpy(y).long().to(DEVICE); X0 = torch.from_numpy(X).float().to(DEVICE); Xa = X0.clone()
    for _ in range(n_iter):
        Xa.requires_grad_(True)
        g = torch.autograd.grad(F.cross_entropy(model(Xa), yt), Xa)[0]
        Xa = (Xa.detach() + alpha * m * g.sign()); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
    return Xa.detach().cpu().numpy()

masks = {'all': None, 'cyber_only': np.isin(np.arange(N_FEAT), CYBER_IDX), 'physical_only': ~np.isin(np.arange(N_FEAT), CYBER_IDX)}
atk_mask = (y_test != 0)
def attack_metrics(pred):
    return {'overall_f1': f1_score(y_test, pred, average='weighted'), 'macro_f1': f1_score(y_test, pred, average='macro'),
            'attack_f1': f1_score(y_test[atk_mask], pred[atk_mask], average='weighted'),
            'evasion_rate': float((pred[atk_mask] == 0).mean()), 'benign_fpr': float((pred[~atk_mask] != 0).mean()),
            'accuracy': float((pred == y_test).mean())}

## 3. Per-seed run: surrogate + two targets, FGSM and PGD sweeps, three threat models

For every seed: train the surrogate, fit the tuned and baseline XGBoost, then sweep epsilon. Each adversarial set is generated **once** (on the surrogate) and scored by both targets.
The tuned target's clean F1 is compared with the NB2 record as a reproducibility check.

In [ ]:
if SMOKE_TEST:
    EPSILONS = [0.0, 0.1, 0.3]
SWEEP = {}      # (seed, method, group, target) -> list of dicts per eps
SEED_INFO = {}; DETAIL = {}
t_all = time.time()
for seed in SEEDS:
    t0 = time.time()
    sur = train_surrogate(seed, X_train, y_train)
    with torch.no_grad(): sur_pred = sur(torch.from_numpy(X_test).float().to(DEVICE)).argmax(1).cpu().numpy()
    tgt = {}
    for tname, p in [('tuned', best_params), ('baseline', BASELINE_PARAMS)]:
        m = xgb.XGBClassifier(**p, **COMMON, random_state=seed); m.fit(X_train, y_train, verbose=False); tgt[tname] = m
    clean = {k: tgt[k].predict(X_test) for k in tgt}
    SEED_INFO[seed] = {'sur_test_f1': f1_score(y_test, sur_pred, average='weighted'),
                       'tuned_clean_f1': f1_score(y_test, clean['tuned'], average='weighted'),
                       'baseline_clean_f1': f1_score(y_test, clean['baseline'], average='weighted'),
                       'agreement_tuned': float((sur_pred == clean['tuned']).mean()), 'agreement_baseline': float((sur_pred == clean['baseline']).mean())}
    torch.save(sur.state_dict(), f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt')
    for method, fn in [('FGSM', fgsm_attack), ('PGD', pgd_attack)]:
        for gname, mk in masks.items():
            if method == 'FGSM' and gname != 'all': continue          # FGSM only for the all-feature threat model
            for tname in tgt: SWEEP[(seed, method, gname, tname)] = []
            for eps in EPSILONS:
                Xa = X_test.copy() if eps == 0.0 else fn(sur, X_test, y_test, eps, mk)
                for tname in tgt:
                    pred = tgt[tname].predict(Xa) if eps > 0 else clean[tname]
                    SWEEP[(seed, method, gname, tname)].append({'epsilon': eps, **attack_metrics(pred)})
                if eps == EPS_DETAIL:
                    for tname in tgt: DETAIL[(seed, method, gname, tname)] = (tgt[tname].predict(Xa) if eps > 0 else clean[tname])
                    if method == 'PGD' and gname == 'all':
                        DETAIL[(seed, 'Xadv')] = Xa.astype(np.float32) if seed == SEEDS[0] else None
    SEED_INFO[seed]['clean_pred_tuned'] = clean['tuned']
    si = SEED_INFO[seed]
    print(f'seed={seed:<5} surrogate F1={si["sur_test_f1"]:.4f} | tuned clean {si["tuned_clean_f1"]:.4f} (agree {si["agreement_tuned"]:.3f}) | '
          f'baseline clean {si["baseline_clean_f1"]:.4f} (agree {si["agreement_baseline"]:.3f}) | {time.time() - t0:.0f}s', flush=True)
    del sur, tgt
print(f'Total {time.time() - t_all:.0f}s')
tuned_clean = np.array([SEED_INFO[s]['tuned_clean_f1'] for s in SEEDS])
print(f'\nTuned clean F1 here: {tuned_clean.mean():.4f} +/- {tuned_clean.std(ddof=1):.4f} | NB2 record: {nb2_tuned.mean():.4f} (difference {tuned_clean.mean() - nb2_tuned.mean():+.4f}; same backend => should be ~0)')

## 4. Epsilon sweep -- mean +/- std over 5 seeds (tuned target, all-feature threat model)

In [ ]:
T4 = stats.t.ppf(0.975, df=len(SEEDS) - 1)
def agg(method, group, target, key):
    a = np.array([[r[key] for r in SWEEP[(s, method, group, target)]] for s in SEEDS]); return a.mean(0), a.std(0, ddof=1), a
rows = []
for eps_i, eps in enumerate(EPSILONS):
    row = {'eps': eps}
    for method in ['FGSM', 'PGD']:
        for key, tag in [('overall_f1', 'F1'), ('evasion_rate', 'Evasion')]:
            m, s, _ = agg(method, 'all', 'tuned', key); row[f'{method} {tag}'] = f'{m[eps_i]:.4f} +/- {s[eps_i]:.4f}'
    rows.append(row)
sweep_table = pd.DataFrame(rows); print(sweep_table.to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
for method, mk in [('FGSM', 'o'), ('PGD', 's')]:
    for ai, key in enumerate(['overall_f1', 'evasion_rate']):
        m, s, _ = agg(method, 'all', 'tuned', key); ax[ai].errorbar(EPSILONS, m, yerr=s, marker=mk, capsize=3, label=method)
for gname, mk in [('all', 's'), ('cyber_only', '^'), ('physical_only', 'v')]:
    m, s, _ = agg('PGD', gname, 'tuned', 'overall_f1'); ax[2].errorbar(EPSILONS, m, yerr=s, marker=mk, capsize=3, label=f'PGD {gname}')
ax[0].set_title('Weighted F1 vs eps (tuned XGBoost, mean +/- sd, 5 seeds)'); ax[1].set_title('Evasion rate (attack -> benign)'); ax[2].set_title('PGD by perturbable feature group')
for a in ax: a.set_xlabel('epsilon (standardised units)'); a.grid(alpha=.3); a.legend()
ax[0].set_ylabel('Overall test F1'); ax[1].set_ylabel('Evasion rate'); ax[2].set_ylabel('Overall test F1')
plt.tight_layout(); plt.savefig(f'attack1_f1_degradation_{PIPELINE_VERSION}.png', dpi=150); plt.show()

## 5. Threat-model ablation at eps = 0.1: which feature group can an attacker use?

This is the answer to "physics immunity versus threat model". Same surrogate, same eps; only the set of perturbable features changes.

In [ ]:
rows = []
for gname in ['all', 'cyber_only', 'physical_only']:
    for tname in ['tuned', 'baseline']:
        i = EPSILONS.index(EPS_DETAIL)
        f1s = np.array([SWEEP[(s, 'PGD', gname, tname)][i]['overall_f1'] for s in SEEDS]); ev = np.array([SWEEP[(s, 'PGD', gname, tname)][i]['evasion_rate'] for s in SEEDS])
        rows.append({'PGD features': gname, 'target': tname, 'n_feat': int(N_FEAT if gname == 'all' else (len(CYBER_IDX) if gname == 'cyber_only' else len(PHYS_IDX))),
                     'F1 under attack': f'{f1s.mean():.4f} +/- {f1s.std(ddof=1):.4f}', 'F1 drop': f'{(np.mean([SEED_INFO[s][tname + "_clean_f1"] for s in SEEDS]) - f1s.mean()):.4f}',
                     'Evasion': f'{ev.mean():.4f} +/- {ev.std(ddof=1):.4f}', '95% CI evasion': f'[{ev.mean() - T4 * ev.std(ddof=1) / np.sqrt(5):.4f}, {ev.mean() + T4 * ev.std(ddof=1) / np.sqrt(5):.4f}]'})
threat_table = pd.DataFrame(rows); print(threat_table.to_string(index=False))
print('\nReading guide: a large drop under cyber_only with a small drop under physical_only means the detector relies on network features that a network attacker can alter, '
      'while physical features carry little standalone signal (NB1b: physical-only is near chance under temporal splits). "Physics immunity" therefore cannot be claimed '
      'from the all-feature numbers alone.')

## 6. Per-class breakdown at eps = 0.1 (PGD, all features) -- 5-seed mean +/- std

In [ ]:
def per_class_evasion(method, group, tname):
    out = np.zeros((len(SEEDS), 5)); 
    for si, s in enumerate(SEEDS):
        p = DETAIL[(s, method, group, tname)]
        for c in range(1, 6): out[si, c - 1] = (p[y_test == c] == 0).mean()
    return out
pc = {}
for method in ['PGD', 'FGSM']:
    for tname in ['tuned', 'baseline']:
        a = per_class_evasion(method, 'all', tname); pc[(method, tname)] = a
print('Per-class evasion rate (true class -> predicted benign), eps=0.1, mean +/- sd over seeds')
print(f"{'class':<14}" + ''.join(f'{m + "/" + t:>22}' for (m, t) in pc))
for ci, cn in enumerate(class_names[1:]):
    print(f'{cn:<14}' + ''.join(f'{pc[k][:, ci].mean():>13.4f} +/- {pc[k][:, ci].std(ddof=1):.4f}' for k in pc))
print('\nPer-class F1 under PGD eps=0.1 (tuned target, mean over seeds) vs clean:')
f1_adv = np.mean([f1_score(y_test, DETAIL[(s, 'PGD', 'all', 'tuned')], average=None, labels=range(6), zero_division=0) for s in SEEDS], 0)
f1_cln = np.mean([f1_score(y_test, SEED_INFO[s]['clean_pred_tuned'], average=None, labels=range(6), zero_division=0) for s in SEEDS], 0)
print(pd.DataFrame({'clean': f1_cln, 'PGD eps0.1': f1_adv, 'drop': f1_cln - f1_adv}, index=class_names).round(4).to_string())
print('\nPer-class evasion under CYBER-ONLY PGD (tuned):')
a = per_class_evasion('PGD', 'cyber_only', 'tuned'); print(pd.Series(a.mean(0), index=class_names[1:]).round(4).to_string())
print('\nSeed-42 classification report, PGD eps=0.1, tuned target:')
print(classification_report(y_test, DETAIL[(42, 'PGD', 'all', 'tuned')], target_names=class_names, digits=4))

## 7. Statistical validation

* **7a** PGD vs FGSM and tuned vs baseline target: paired t-tests across the 5 seeds (n=5; Wilcoxon cannot go below p=0.0625 here).
* **7b** McNemar (clean vs adversarial correctness) per seed. With ~46,000 discordant pairs it will be "significant" for any effect; reported for completeness and because it is paired.
* **7c** Evasion-rate CI at eps=0.1 by **burst-clustered bootstrap** (resampling the test bursts, 2000 reps) next to the naive row bootstrap. The row bootstrap treats 229 identical rows as independent and understates the uncertainty; the clustered one is the defensible interval.

In [ ]:
i = EPSILONS.index(EPS_DETAIL)
get = lambda m, g, t, k: np.array([SWEEP[(s, m, g, t)][i][k] for s in SEEDS])
print('--- 7a. paired tests over seeds (eps=0.1) ---')
stat_tests = {}
for key in ['evasion_rate', 'overall_f1']:
    for name, a, b in [('PGD vs FGSM (tuned)', get('PGD', 'all', 'tuned', key), get('FGSM', 'all', 'tuned', key)),
                       ('tuned vs baseline target (PGD)', get('PGD', 'all', 'tuned', key), get('PGD', 'all', 'baseline', key)),
                       ('cyber-only vs all (PGD, tuned)', get('PGD', 'cyber_only', 'tuned', key), get('PGD', 'all', 'tuned', key))]:
        d = a - b; tt = stats.ttest_rel(a, b) if d.std(ddof=1) > 0 else None
        stat_tests[(key, name)] = {'mean_diff': float(d.mean()), 'p': float(tt.pvalue) if tt else 1.0, 'per_seed': d.tolist()}
        print(f'{key:<13} {name:<34} diff={d.mean():+.4f} (sd {d.std(ddof=1):.4f})  p={stat_tests[(key, name)]["p"]:.4f}')

def mcnemar(y_true, p_clean, p_adv):
    cc, ca = p_clean == y_true, p_adv == y_true; b = int((cc & ~ca).sum()); c = int((~cc & ca).sum()); n = b + c
    if n == 0: return {'b': b, 'c': c, 'p_value': 1.0, 'method': 'degenerate'}
    if n < 25: return {'b': b, 'c': c, 'p_value': float(stats.binomtest(min(b, c), n, .5).pvalue), 'method': 'exact'}
    st = (abs(b - c) - 1) ** 2 / n; return {'b': b, 'c': c, 'statistic': float(st), 'p_value': float(stats.chi2.sf(st, 1)), 'method': 'chi-square (cc)'}
print('\n--- 7b. McNemar per seed (tuned target, PGD eps=0.1, all features) ---')
mc = {}
for s in SEEDS:
    mc[s] = mcnemar(y_test, SEED_INFO[s]['clean_pred_tuned'], DETAIL[(s, 'PGD', 'all', 'tuned')])
    print(f'  seed {s:<5} b={mc[s]["b"]:>6} c={mc[s]["c"]:>6} p={mc[s]["p_value"]:.3g}')

print('\n--- 7c. evasion-rate CI, PGD eps=0.1, tuned target (pooled over 5 seeds) ---')
rng = np.random.RandomState(42); nb_ = test_burst.max() + 1
n_atk = np.bincount(test_burst[atk_mask], minlength=nb_).astype(float)
boot_cluster = np.zeros((len(SEEDS), 2000)); naive = []
for si, s in enumerate(SEEDS):
    ev = np.bincount(test_burst[atk_mask], weights=(DETAIL[(s, 'PGD', 'all', 'tuned')][atk_mask] == 0).astype(float), minlength=nb_)
    idx = rng.randint(0, nb_, size=(2000, nb_))
    boot_cluster[si] = ev[idx].sum(1) / np.maximum(n_atk[idx].sum(1), 1)
    pa = DETAIL[(s, 'PGD', 'all', 'tuned')][atk_mask]; r2 = rng.randint(0, len(pa), size=(300, len(pa)))
    naive.append(np.percentile((pa[r2] == 0).mean(1), [2.5, 97.5]))
pooled = boot_cluster.mean(0); pt = get('PGD', 'all', 'tuned', 'evasion_rate').mean()
lo, hi = np.percentile(pooled, [2.5, 97.5])
nlo, nhi = np.mean(naive, 0)
print(f'  evasion point estimate (mean over seeds) {pt:.4f}')
print(f'  burst-clustered 95% CI (mean of 5 seeds): [{lo:.4f}, {hi:.4f}] width {hi - lo:.4f}')
print(f'  naive row bootstrap  95% CI (seed-averaged): [{nlo:.4f}, {nhi:.4f}] width {nhi - nlo:.4f}  <- too narrow, shown for contrast')
print(f'  effective independent units: {int(len(np.unique(test_burst[atk_mask])))} attack bursts vs {int(atk_mask.sum())} rows')

## 8. Save (Defence 1 / later notebooks read these)

In [ ]:
os.makedirs('.', exist_ok=True)
clean_info = {s: {k: v for k, v in SEED_INFO[s].items() if k != 'clean_pred_tuned'} for s in SEEDS}
joblib.dump({
    'PIPELINE_VERSION': PIPELINE_VERSION, 'BACKEND': BACKEND_NAME, 'SEEDS': SEEDS, 'EPSILONS': EPSILONS, 'EPS_DETAIL': EPS_DETAIL, 'PGD_ITERS': PGD_ITERS,
    'target': 'Optuna-tuned XGBoost (NB2 v6) + untuned baseline on same backend', 'tuned_params': best_params,
    'sweep': {f'{k[0]}|{k[1]}|{k[2]}|{k[3]}': v for k, v in SWEEP.items()},
    'seed_info': clean_info, 'sweep_table': sweep_table, 'threat_model_table': threat_table,
    'per_class_evasion_eps01': {f'{m}|{t}': pc[(m, t)] for (m, t) in pc}, 'class_names': class_names,
    'stat_tests': {f'{k[0]}|{k[1]}': v for k, v in stat_tests.items()}, 'mcnemar': mc,
    'evasion_ci_cluster': {'point': float(pt), 'lo': float(lo), 'hi': float(hi), 'naive_lo': float(nlo), 'naive_hi': float(nhi)},
    'perturbable': {'all': N_FEAT, 'cyber_only': int(len(CYBER_IDX)), 'physical_only': int(len(PHYS_IDX))}, 'constant_features': [FEATURE_COLS[i] for i in const_idx],
    # legacy keys read by earlier-style downstream notebooks (seed 42, tuned target, all features)
    'results_pgd': SWEEP[(42, 'PGD', 'all', 'tuned')], 'results_fgsm': SWEEP[(42, 'FGSM', 'all', 'tuned')],
    'clean_test_f1': SEED_INFO[42]['tuned_clean_f1'], 'surrogate_target_agreement': SEED_INFO[42]['agreement_tuned'],
}, f'attack1_results_{PIPELINE_VERSION}.pkl')
np.save(f'X_adv_pgd_eps01_{PIPELINE_VERSION}.npy', DETAIL[(SEEDS[0], 'Xadv')])
sweep_table.to_csv(f'attack1_sweep_{PIPELINE_VERSION}.csv', index=False); threat_table.to_csv(f'attack1_threat_model_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB3', 'pipeline_version': PIPELINE_VERSION, 'backend': BACKEND_NAME, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'python': platform.python_version(),
           'seeds': SEEDS, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb3_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack1_results_{PIPELINE_VERSION}.pkl', f'X_adv_pgd_eps01_{PIPELINE_VERSION}.npy', f'attack1_sweep_{PIPELINE_VERSION}.csv', f'attack1_threat_model_{PIPELINE_VERSION}.csv',
          f'attack1_f1_degradation_{PIPELINE_VERSION}.png', f'nb3_run_manifest_{PIPELINE_VERSION}.json'] + [f'surrogate_mlp_{PIPELINE_VERSION}_seed{s}.pt' for s in SEEDS]:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)